In [9]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from scipy.integrate import solve_ivp #(initial value problem)


rng = np.random.default_rng(seed=42) # random number generator


N_SPECIES   = 20
N_RESOURCES = 4
N_GENES     = 6
N_BROTHS    = 8

RESOURCE_NAMES = [f"c{k+1}" for k in range(N_RESOURCES)]
SPECIES_NAMES  = [f"Species_{i+1:02d}" for i in range(N_SPECIES)]
BROTH_NAMES    = [f"Broth_{b+1:02d}"   for b in range(N_BROTHS)]

OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads"

In [10]:
def build_species_traits():
    """
    Generate binary genomes and uptake rates for all species.

    Binary genome: genes[i, g] in {0, 1}
      Each gene = whether species i has the enzyme for a metabolic task.

    Uptake rates: c_matrix[i, alpha] in (0, 1)
      How fast species i can consume nutrient alpha.
      Computed as: sigmoid( genes @ W + bias )
      - W  is a random gene-to-nutrient weight matrix
      - bias makes most species specialists (low affinity for most nutrients)
      - sigmoid squashes values into (0,1) range

    Returns
    -------
    genes    : shape (N_SPECIES, N_GENES)    — binary
    c_matrix : shape (N_SPECIES, N_RESOURCES) — uptake rates in (0,1)
    """
    genes = rng.integers(0, 2, size=(N_SPECIES, N_GENES))
    # rng.integers(low,high, size=..) draws random integers in the range [low,high)
    W     = rng.normal(loc=0.0, scale=0.5, size=(N_GENES, N_RESOURCES))
    # W represents how much gene g contributes to the ability to consume nutrient alpha
    bias  = rng.uniform(-1.5, -0.5, size=N_RESOURCES)
    raw   = genes.astype(float) @ W + bias
    # @ performs matrix multiplication in python
    c_matrix = 1.0 / (1.0 + np.exp(-raw))
    return genes, c_matrix

In [11]:
def build_broths():
    """
    Generate chemostat broth environments.

    Each broth is now defined by:
      delta   : dilution rate (h^-1). This is the rate at which liquid flows
                through the chemostat. Typical range: 0.05 to 0.2 h^-1.
                The same delta applies to both nutrient and cell dilution.
      R0      : nutrient concentrations in the INCOMING FRESH MEDIA (feed bottle).
                When liquid flows in, it brings nutrients at concentration R0.
                The dominant nutrient is enriched (higher R0).
      w       : energy content of each nutrient (unchanged)
      l       : leakage fraction (unchanged)

    The net nutrient dynamics from dilution alone:
      dR/dt|dilution = delta*R0_alpha - delta*R_alpha = delta*(R0_alpha - R_alpha)

    All species are present in every broth.

    Returns
    -------
    broths : list of dicts
    """
    broths = []
    for b in range(N_BROTHS):
        dominant = int(rng.integers(0, N_RESOURCES))
        # randomly picks which nutrient is the dominant one in this broth


        delta = float(rng.uniform(0.05, 0.20))   # h^-1

        # R0: concentration of each nutrient in the FEED (fresh media)
        # Units: arbitrary concentration units (e.g. mM or g/L)
        # The dominant nutrient has a much higher feed concentration.
        R0 = rng.uniform(0.5, 2.0, size=N_RESOURCES)
        R0[dominant] *= rng.uniform(3.0, 6.0)   # enrich dominant nutrient in feed

        # Energy content of each nutrient (unchanged from before)
        w = rng.uniform(0.6, 1.2, size=N_RESOURCES)

        # Leakage fraction (unchanged from before)
        l = rng.uniform(0.6, 0.9, size=N_RESOURCES)

        members = list(range(N_SPECIES))

        broths.append({
            "name"             : BROTH_NAMES[b],
            "delta"            : delta,    # dilution rate h^-1  (NEW — replaces tau)
            "R0"               : R0,       # feed concentrations  (NEW — replaces kappa)
            "w"                : w,
            "l"                : l,
            "dominant"         : dominant,
            "members"          : members,
        })
    return broths

In [12]:
def build_metabolic_matrix():
    """
    Build metabolic cross-feeding matrix D.
    D[alpha, beta] = fraction of byproducts from eating beta that comes out as alpha.
    Each column of D sums to 1 (byproducts are fully distributed).
    Shape: (N_RESOURCES, N_RESOURCES)
    """
    D = np.zeros((N_RESOURCES, N_RESOURCES))
    for beta in range(N_RESOURCES):
        D[:, beta] = rng.dirichlet(np.ones(N_RESOURCES))
    return D
    

In [13]:
def micrm_ode(t, y, c, w, l, m, g, delta, R0, D):
    """
    Updated MiCRM ODE with explicit chemostat dilution for BOTH species and nutrients.

    Parameters
    ----------
    t     : float   — current time (required by solve_ivp)
    y     : ndarray shape (n_sp + n_res,)
              y[:n_sp]  = N_i  species abundances
              y[n_sp:]  = R_alpha  nutrient concentrations
    c     : ndarray (n_sp, n_res)  — uptake rates
    w     : ndarray (n_res,)       — energy content per nutrient
    l     : ndarray (n_res,)       — leakage fractions
    m     : ndarray (n_sp,)        — maintenance costs
    g     : ndarray (n_sp,)        — growth conversion factors
    delta : float                  — dilution rate h^-1  [NEW parameter]
    R0    : ndarray (n_res,)       — feed nutrient concentrations  [NEW parameter]
    D     : ndarray (n_res, n_res) — cross-feeding matrix

    Returns
    -------
    dydt  : ndarray (n_sp + n_res,) — time derivatives [dN/dt, dR/dt]
    """
    n_sp  = c.shape[0]
    n_res = c.shape[1]

    N = np.maximum(y[:n_sp], 0.0)
    R = np.maximum(y[n_sp:], 0.0)

    effective_w  = w * (1.0 - l)           # energy actually used for growth
    energy_intake = c @ (effective_w * R)  # shape (n_sp,)
    net_growth   = g * (energy_intake - m) # shape (n_sp,)

    dN = N * net_growth - delta * N        

    dilution_net = delta * (R0 - R)        

    # TERM 2: Consumption by all species (unchanged)
    uptake = (c.T @ N) * R                 # shape (n_res,)

    # TERM 3: Cross-feeding (byproducts re-enter nutrient pool — unchanged)
    consumption_flux = (c.T @ N) * R
    leaked_flux      = l * consumption_flux
    eps              = 1e-12
    crossfeed        = D @ (w * leaked_flux) / (w + eps)   # shape (n_res,)

    dR = dilution_net - uptake + crossfeed  # UPDATED: dilution_net replaces old terms

    return np.concatenate([dN, dR])

In [14]:
def run_micrm(c_full, w, l, m_full, g_full, delta, R0, D,
              species_subset, t_end=500.0):
    """
    Integrate the updated MiCRM ODE forward to approximate steady state.
    This function calls solve_ivp to integrate the ODEs forward in time until the system
    reaches steady state (abundances and nutrient levels stop changing)

    Parameters
    ----------
    c_full        : ndarray (N_SPECIES, N_RESOURCES) uptake matrix for all 20 species
    w             : ndarray (N_RESOURCES,)
    l             : ndarray (N_RESOURCES,)
    m_full        : ndarray (N_SPECIES,)
    g_full        : ndarray (N_SPECIES,)
    delta         : float   — dilution rate h^-1          [NEW]
    R0            : ndarray (N_RESOURCES,) — feed concentrations  [NEW]
    D             : ndarray (N_RESOURCES, N_RESOURCES)
    species_subset: list of int
    t_end         : float   — integration end time (hours)

    Returns
    -------
    N_final : ndarray (N_SPECIES,) — final abundances (zeros for absent species)
    R_final : ndarray (N_RESOURCES,) — final nutrient concentrations
    sol     : ODE solution object
    """
    c_sub = c_full[species_subset, :]
    m_sub = m_full[species_subset]
    g_sub = g_full[species_subset]
    n_sub = len(species_subset)

    # Initial conditions:
    # Species start at a small equal abundance.
    # Nutrients start at R0 — the feed concentration, which is what you'd have
    # in an empty chemostat (no microbes consuming nutrients yet).
    N_init = np.full(n_sub, 0.01) # all species start at the same low abundance
    R_init = R0.copy() # nutrients start at the feed concentration
    y0     = np.concatenate([N_init, R_init])

    sol = solve_ivp(
        fun    = micrm_ode,
        t_span = (0.0, t_end),
        y0     = y0,
        args   = (c_sub, w, l, m_sub, g_sub, delta, R0, D),  # UPDATED args
        method = "RK45",
        t_eval = np.linspace(0, t_end, 600),
        rtol   = 1e-6,
        atol   = 1e-8,
    )

    y_final = sol.y[:, -1]
    N_ss    = np.maximum(y_final[:n_sub], 0.0)
    R_final = np.maximum(y_final[n_sub:], 0.0)

    N_final = np.zeros(N_SPECIES)
    for local_idx, global_idx in enumerate(species_subset):
        N_final[global_idx] = N_ss[local_idx]

    return N_final, R_final, sol

In [15]:
def suppression_score(N_comm, N_solo, noise_std=0.02):
    """
    suppression_score = 1 - (N_community / N_solo)

    0 = no suppression (thrives as well in community as alone)
    1 = completely eliminated by competition
    """
    eps = 1e-10
    if N_solo < eps:
        return 0.0
    raw   = 1.0 - (N_comm / (N_solo + eps))
    score = float(np.clip(raw, 0.0, 1.0))
    noise = float(rng.normal(0.0, noise_std))
    return float(np.clip(score + noise, 0.0, 1.0))

In [16]:
def generate_dataset():
    """
    Full pipeline: traits → ODE → suppression scores → DataFrame.
    """

    print("\n[1] Generating species traits...")
    genes, c_matrix = build_species_traits()
    m = rng.uniform(0.2, 0.6, size=N_SPECIES)
    g = rng.uniform(0.8, 1.2, size=N_SPECIES)
    print(f"    {N_SPECIES} species, {N_GENES} genes, {N_RESOURCES} nutrients")
    print(f"    Uptake rate range: [{c_matrix.min():.3f}, {c_matrix.max():.3f}]")

    print("\n[2] Building metabolic cross-feeding matrix D...")
    D = build_metabolic_matrix()

    print("\n[3] Generating broth environments...")
    broths = build_broths()
    for b in broths:
        dominant_name = RESOURCE_NAMES[b["dominant"]]
        print(f"    {b['name']}: dominant={dominant_name}, "
              f"delta={b['delta']:.3f} h^-1, R0_max={b['R0'].max():.2f}")

    print("\n[4] Running MiCRM ODE (community + solo) for each broth...")
    rows = []

    for broth in broths:
        members = broth["members"]
        delta   = broth["delta"]
        R0      = broth["R0"]
        w       = broth["w"]
        l       = broth["l"]

        # Community run
        N_comm, R_ss, _ = run_micrm(
            c_full=c_matrix, w=w, l=l, m_full=m, g_full=g,
            delta=delta, R0=R0, D=D,
            species_subset=members, t_end=500.0,
        )

        # Solo runs
        N_solo = np.zeros(N_SPECIES)
        for sp_idx in members:
            N_s, _, _ = run_micrm(
                c_full=c_matrix, w=w, l=l, m_full=m, g_full=g,
                delta=delta, R0=R0, D=D,
                species_subset=[sp_idx], t_end=400.0,
            )
            N_solo[sp_idx] = N_s[sp_idx]

        broth_scores = []
        for sp_idx in members:
            score = suppression_score(N_comm[sp_idx], N_solo[sp_idx])
            broth_scores.append(score)

            row = {
                "species_id"        : SPECIES_NAMES[sp_idx],
                "broth_id"          : broth["name"],
                "dominant_nutrient" : RESOURCE_NAMES[broth["dominant"]],
                "delta_per_h"       : round(broth["delta"], 4),   # UPDATED column name
                "N_solo"            : round(float(N_solo[sp_idx]), 4),
                "N_community"       : round(float(N_comm[sp_idx]), 4),
                "suppression_score" : round(score, 4),
            }
            for g_idx in range(N_GENES):
                row[f"gene_{g_idx+1}"] = int(genes[sp_idx, g_idx])
            for k in range(N_RESOURCES):
                row[f"uptake_{RESOURCE_NAMES[k]}"] = round(float(c_matrix[sp_idx, k]), 4)
            row["maintenance_cost"] = round(float(m[sp_idx]), 4)
            rows.append(row)

        print(f"    {broth['name']}: scores [{min(broth_scores):.3f}, "
              f"{max(broth_scores):.3f}], mean={np.mean(broth_scores):.3f}")

    df = pd.DataFrame(rows)
    gene_cols   = [f"gene_{g+1}" for g in range(N_GENES)]
    uptake_cols = [f"uptake_{r}" for r in RESOURCE_NAMES]
    col_order   = (
        ["species_id", "broth_id", "dominant_nutrient", "delta_per_h",
         "maintenance_cost"]
        + gene_cols + uptake_cols
        + ["N_solo", "N_community", "suppression_score"]
    )
    df = df[col_order]

    print(f"\n  Dataset shape: {df.shape}")
    print(f"  Score summary:\n{df['suppression_score'].describe().round(3)}")
    return df, genes, c_matrix, m, g, broths, D

In [17]:
def make_figures(df, genes, c_matrix, m, g, broths, D):
    print("\n[5] Generating figures...")

    # ── FIGURE 1: ODE Time Series ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    b0 = broths[0]

    c_sub = c_matrix[b0["members"], :]
    m_sub = m[b0["members"]]
    g_sub = g[b0["members"]]
    N_init = np.full(N_SPECIES, 0.01)
    R_init = b0["R0"].copy()            # UPDATED: start at R0
    y0 = np.concatenate([N_init, R_init])

    sol = solve_ivp(
        micrm_ode, (0, 500), y0,
        args=(c_sub, b0["w"], b0["l"], m_sub, g_sub,
              b0["delta"], b0["R0"], D),  # UPDATED args
        t_eval=np.linspace(0, 500, 800),
        method="RK45", rtol=1e-6, atol=1e-8,
    )

    colors_sp = plt.cm.tab20(np.linspace(0, 1, N_SPECIES))
    for i in range(N_SPECIES):
        axes[0].plot(sol.t, sol.y[i], color=colors_sp[i], linewidth=1, alpha=0.7)
    axes[0].set_xlabel("Time (hours)")
    axes[0].set_ylabel("Species abundance N_i")
    axes[0].set_title(
        f"Species Dynamics — {b0['name']}\n"
        f"(delta={b0['delta']:.3f} h^-1, with cell washout -delta*N)",
        fontweight="bold")
    axes[0].set_yscale("symlog", linthresh=0.001)
    axes[0].grid(True, alpha=0.3)

    res_colors = plt.cm.Set1(np.linspace(0, 1, N_RESOURCES))
    for k in range(N_RESOURCES):
        axes[1].plot(sol.t, sol.y[N_SPECIES + k],
                     color=res_colors[k], linewidth=2.5, label=RESOURCE_NAMES[k])
    axes[1].set_xlabel("Time (hours)")
    axes[1].set_ylabel("Nutrient concentration R_alpha")
    axes[1].set_title(
        "Nutrient Dynamics\n"
        r"$dR/dt = \delta(R^0-R) - \mathrm{uptake} + \mathrm{crossfeed}$",
        fontweight="bold")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    fig.suptitle(
        f"Figure 1: Updated MiCRM Dynamics [{b0['name']}]\n"
        r"Species: $\frac{dN_i}{dt}=g_i N_i[\ldots]-\delta N_i$  |  "
        r"Nutrients: $\frac{dR_\alpha}{dt}=\delta(R^0_\alpha-R_\alpha)-\ldots$",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig1_ode_dynamics.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    -> fig1_ode_dynamics.png")

    # ── FIGURE 2: Genome and Uptake Rate Heatmaps ─────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    fig.suptitle("Figure 2: Genome -> Uptake Rate Mapping", fontsize=12, fontweight="bold")
    sns.heatmap(genes.astype(float), ax=axes[0], cmap="Blues", vmin=0, vmax=1,
                xticklabels=[f"G{g+1}" for g in range(N_GENES)],
                yticklabels=[f"Sp{i+1}" for i in range(N_SPECIES)],
                linewidths=0.3, linecolor="white",
                cbar_kws={"label": "Gene state (0/1)"})
    axes[0].set_title("Binary Genomes", fontweight="bold")
    axes[0].tick_params(axis="y", labelsize=7)

    sns.heatmap(c_matrix, ax=axes[1], cmap="YlOrRd", vmin=0, vmax=1,
                xticklabels=RESOURCE_NAMES,
                yticklabels=[f"Sp{i+1}" for i in range(N_SPECIES)],
                linewidths=0.3, linecolor="white",
                cbar_kws={"label": "Uptake rate c_i_alpha"})
    axes[1].set_title("Uptake Rates c_i_alpha (from genome)", fontweight="bold")
    axes[1].tick_params(axis="y", labelsize=7)

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig2_genome_uptake.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    -> fig2_genome_uptake.png")

    # ── FIGURE 3: Cross-Feeding Matrix + Feed Concentrations ──────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(
        "Figure 3: Cross-Feeding Matrix D and Feed Concentrations R0\n"
        "(R0 replaces kappa as the nutrient input parameter)",
        fontsize=12, fontweight="bold"
    )
    sns.heatmap(D, ax=axes[0], cmap="Blues", vmin=0, vmax=1,
                xticklabels=[f"Input {r}" for r in RESOURCE_NAMES],
                yticklabels=[f"Output {r}" for r in RESOURCE_NAMES],
                annot=True, fmt=".2f", linewidths=0.5,
                cbar_kws={"label": "Fraction of byproducts"})
    axes[0].set_title("D matrix: Cross-Feeding", fontweight="bold")

    R0_arr = np.array([b["R0"] for b in broths])
    sns.heatmap(R0_arr, ax=axes[1], cmap="YlOrRd",
                xticklabels=RESOURCE_NAMES,
                yticklabels=[b["name"].replace("Broth_", "B") for b in broths],
                annot=True, fmt=".1f", linewidths=0.5,
                cbar_kws={"label": "Feed concentration R0"})
    axes[1].set_title(
        r"Feed Concentrations $R^0_\alpha$ per Broth" + "\n"
        r"(inflow = $\delta \cdot R^0_\alpha$)",
        fontweight="bold")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig3_crossfeeding_supply.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    -> fig3_crossfeeding_supply.png")

    # ── FIGURE 4: Solo vs Community Abundances ────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Figure 4: Solo vs Community Steady-State Abundances",
                 fontsize=12, fontweight="bold")
    mean_df = (df.groupby("species_id")[["N_solo", "N_community", "suppression_score"]]
               .mean().reset_index().sort_values("N_solo", ascending=False))

    x = np.arange(len(mean_df))
    axes[0].bar(x - 0.2, mean_df["N_solo"],      width=0.38, label="Solo",
                color="#2196F3", alpha=0.8)
    axes[0].bar(x + 0.2, mean_df["N_community"], width=0.38, label="Community",
                color="#FF5722", alpha=0.8)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(
        [s.replace("Species_", "S") for s in mean_df["species_id"]],
        rotation=70, fontsize=7)
    axes[0].set_ylabel("Mean steady-state abundance")
    axes[0].set_title("Solo vs Community\n(averaged over all broths)", fontweight="bold")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis="y")

    sc = axes[1].scatter(df["N_solo"], df["N_community"],
                          c=df["suppression_score"], cmap="RdYlGn_r",
                          s=40, alpha=0.6, edgecolors="none")
    lim = max(df["N_solo"].max(), df["N_community"].max()) * 1.05
    axes[1].plot([0, lim], [0, lim], "k--", lw=1.5, label="No suppression line")
    plt.colorbar(sc, ax=axes[1], label="Suppression score")
    axes[1].set_xlabel("N_solo")
    axes[1].set_ylabel("N_community")
    axes[1].set_title("All data (colour = suppression)\nBelow diagonal = suppressed",
                       fontweight="bold")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig4_solo_vs_community.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    -> fig4_solo_vs_community.png")

    # ── FIGURE 5: Dataset Summary ─────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle("Figure 5: MiCRM Dataset Summary", fontsize=12, fontweight="bold")

    axes[0,0].hist(df["suppression_score"], bins=25,
                   color="#3F51B5", edgecolor="white", alpha=0.85)
    axes[0,0].axvline(df["suppression_score"].mean(), color="red",
                       linestyle="--", lw=2,
                       label=f"Mean = {df['suppression_score'].mean():.3f}")
    axes[0,0].set_xlabel("Suppression score")
    axes[0,0].set_ylabel("Count")
    axes[0,0].set_title("Distribution of Suppression Scores", fontweight="bold")
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)

    dn_colors = plt.cm.Set2(np.linspace(0, 1, N_RESOURCES))
    for k, rname in enumerate(RESOURCE_NAMES):
        subset = df[df["dominant_nutrient"] == rname]["suppression_score"]
        if len(subset) > 0:
            axes[0,1].hist(subset, bins=12, alpha=0.55,
                           label=f"{rname} (n={len(subset)})",
                           color=dn_colors[k], edgecolor="none")
    axes[0,1].set_xlabel("Suppression score")
    axes[0,1].set_ylabel("Count")
    axes[0,1].set_title("Scores by Dominant Nutrient", fontweight="bold")
    axes[0,1].legend(fontsize=8)
    axes[0,1].grid(True, alpha=0.3)

    pivot = df.pivot_table(index="species_id", columns="broth_id",
                            values="suppression_score", aggfunc="mean")
    pivot.index   = [s.replace("Species_", "S") for s in pivot.index]
    pivot.columns = [c.replace("Broth_", "B") for c in pivot.columns]
    sns.heatmap(pivot, ax=axes[1,0], cmap="YlOrRd", vmin=0, vmax=1,
                linewidths=0.3, linecolor="lightgray",
                cbar_kws={"label": "Suppression score"})
    axes[1,0].set_title("Species x Broth Heatmap", fontweight="bold")
    axes[1,0].tick_params(axis="y", labelsize=7)

    mean_supp = df.groupby("species_id")["suppression_score"].mean()
    sp_m = [m[i] for i in range(N_SPECIES)]
    axes[1,1].scatter(sp_m, mean_supp.values, alpha=0.7, s=60, color="#9C27B0")
    axes[1,1].set_xlabel("Maintenance cost m_i")
    axes[1,1].set_ylabel("Mean suppression score")
    axes[1,1].set_title("Maintenance Cost vs Suppression", fontweight="bold")
    axes[1,1].grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/fig5_dataset_summary.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("    -> fig5_dataset_summary.png")


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    df, genes, c_matrix, m, g, broths, D = generate_dataset()

    out_path = f"{OUTPUT_DIR}/micrm_dataset.csv"
    df.to_csv(out_path, index=False)
    print(f"\n[Done] Dataset saved -> micrm_dataset.csv  shape: {df.shape}")
    print(f"       Columns: {list(df.columns)}")

    make_figures(df, genes, c_matrix, m, g, broths, D)
    print("\n[Done] All figures saved.")

    print("\n[Sample rows]:")
    print(df.head(10).to_string(index=False))


[1] Generating species traits...
    20 species, 6 genes, 4 nutrients
    Uptake rate range: [0.071, 0.559]

[2] Building metabolic cross-feeding matrix D...

[3] Generating broth environments...
    Broth_01: dominant=c3, delta=0.090 h^-1, R0_max=6.35
    Broth_02: dominant=c2, delta=0.165 h^-1, R0_max=8.74
    Broth_03: dominant=c4, delta=0.075 h^-1, R0_max=9.64
    Broth_04: dominant=c4, delta=0.162 h^-1, R0_max=3.76
    Broth_05: dominant=c2, delta=0.072 h^-1, R0_max=2.84
    Broth_06: dominant=c3, delta=0.069 h^-1, R0_max=4.75
    Broth_07: dominant=c3, delta=0.195 h^-1, R0_max=9.14
    Broth_08: dominant=c2, delta=0.140 h^-1, R0_max=5.95

[4] Running MiCRM ODE (community + solo) for each broth...
    Broth_01: scores [0.000, 1.000], mean=0.497
    Broth_02: scores [0.000, 1.000], mean=0.847
    Broth_03: scores [0.088, 1.000], mean=0.941
    Broth_04: scores [0.000, 0.000], mean=0.000
    Broth_05: scores [0.000, 0.985], mean=0.190
    Broth_06: scores [0.000, 1.000], mean=0.797